# ReClip - Colab Whisper + Diarization API

Bu notebook GPU uzerinde **Whisper large-v3** + **pyannote speaker diarization** calistirir ve ReClip'in cagirabilecegi HTTP endpoint acar.

### Adimlar
1. Uzt menuden **Runtime > Change runtime type > T4 GPU** sec
2. Hucreleri sirayla calistir
3. Son hucredeki `COLAB URL` degerini ReClip Ayarlar > Colab URL alanina yapistir

### Token'lar
- **Ngrok**: [ngrok.com](https://ngrok.com) -> Dashboard > Your Authtoken -> `NGROK_TOKEN`
- **HuggingFace** (pyannote icin): [hf.co/settings/tokens](https://huggingface.co/settings/tokens) -> read token -> `HF_TOKEN`
  - Ayrica [pyannote/speaker-diarization-3.1](https://huggingface.co/pyannote/speaker-diarization-3.1) sayfasinda "Agree and access" butonuna tikla

In [ ]:
NGROK_TOKEN = ""  # ngrok.com'dan al (ucretsiz)
HF_TOKEN    = ""  # huggingface.co read token (pyannote icin)

# Model secimi: 'large-v3' en iyi kalite | 'large-v3-turbo' biraz daha hizli
WHISPER_MODEL = "large-v3"

In [ ]:
!pip install -q faster-whisper fastapi uvicorn python-multipart pyngrok
!pip install -q pyannote.audio==3.3.2

In [ ]:
import torch
from faster_whisper import WhisperModel
from pyannote.audio import Pipeline as DiarizePipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
compute = "float16" if device == "cuda" else "int8"
print(f"Cihaz: {device.upper()} | Compute: {compute}")

if device == "cpu":
    print("UYARI: GPU bulunamadi! Runtime > Change runtime type > T4 GPU sec.")

print(f"Whisper yukleniyor: {WHISPER_MODEL} ...")
model = WhisperModel(WHISPER_MODEL, device=device, compute_type=compute)

diarize_pipe = None
if HF_TOKEN:
    print("pyannote yukleniyor...")
    diarize_pipe = DiarizePipeline.from_pretrained(
        "pyannote/speaker-diarization-3.1",
        use_auth_token=HF_TOKEN,
    )
    if device == "cuda":
        diarize_pipe.to(torch.device("cuda"))
    print("Diarization aktif.")
else:
    print("HF_TOKEN bos -> diarization devre disi. Tek konusmaci modu calisir.")

print("Modeller hazir!")

In [ ]:
import os
import tempfile
import threading
import uvicorn
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import JSONResponse
from pyngrok import ngrok

api = FastAPI(title="ReClip Whisper+Diarize API")


def assign_speakers(whisper_segments, diarize_segments):
    """Her whisper segmentine en cok ortustugu speaker'i ata."""
    out = []
    for w in whisper_segments:
        ws, we = w["start"], w["end"]
        best_spk, best_ov = None, 0.0
        for d in diarize_segments:
            ov = max(0, min(we, d["end"]) - max(ws, d["start"]))
            if ov > best_ov:
                best_ov = ov
                best_spk = d["speaker"]
        out.append({**w, "speaker": best_spk or "SPK0"})
    return out


@api.get("/health")
def health():
    return {"status": "ok", "model": WHISPER_MODEL, "device": device, "diarize": diarize_pipe is not None}


@api.post("/transcribe")
async def transcribe(
    file: UploadFile = File(...),
    src_lang: str = Form("auto"),
    diarize: str = Form("false"),
):
    suffix = os.path.splitext(file.filename or ".mp3")[1] or ".mp3"
    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:
        tmp.write(await file.read())
        tmp_path = tmp.name

    try:
        kwargs = {} if src_lang == "auto" else {"language": src_lang}
        segments_gen, info = model.transcribe(tmp_path, vad_filter=True, beam_size=5, **kwargs)
        segments = [
            {"start": round(s.start, 3), "end": round(s.end, 3), "text": s.text.strip()}
            for s in segments_gen
            if s.text.strip()
        ]

        do_diarize = diarize.lower() in ("true", "1", "yes") and diarize_pipe is not None
        if do_diarize:
            dres = diarize_pipe(tmp_path)
            diar = [
                {"start": round(turn.start, 3), "end": round(turn.end, 3), "speaker": spk}
                for turn, _, spk in dres.itertracks(yield_label=True)
            ]
            segments = assign_speakers(segments, diar)

        return JSONResponse({
            "segments": segments,
            "language": info.language,
            "diarized": do_diarize,
        })
    except Exception as e:
        return JSONResponse({"error": str(e)}, status_code=500)
    finally:
        os.unlink(tmp_path)


if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(8000, bind_tls=True)
print("
" + "=" * 55)
print(f"  COLAB URL: {tunnel.public_url}")
print("=" * 55)
print("Bu URL'yi ReClip > Ayarlar > Colab URL alanina gir.")
print("=" * 55 + "
")

def _run():
    uvicorn.run(api, host="0.0.0.0", port=8000, log_level="warning")
threading.Thread(target=_run, daemon=True).start()
print("Sunucu calisiyor. Notebook acik kaldikca aktif.")